# BDI475 - Class 22: Data Integrity Demo

- Student registration information, including student information, course information and registration information, are stored in one csv file.
- Create relational database tables to host the information. Each table has an autoincrement id as primary key.
    - student_profile: store student information
    - course_profile: store course information
    - registration: store registration information
- Extract data from the original csv file and populate data into the DB tables.
- Add new registration record to the DB.
- Explore tables with DB Browser/DBeaver.

In [4]:
import sqlite3
import pandas as pd

## Student Registration Infomration

- Student_profile columns: net_id, full_name, major
- Course_profile columns: course_no, course_title, department, credits
- Registration columns: student_id(student_profile primary key), course_id(course_profile primary key), term, grade

In [6]:
df = pd.read_csv('student_registration.csv')
df

,net_id,full_name,major,course_no,course_title,department,credits,term,grade
0,aa123,Alice Anderson,CS,CS101,Intro to Computer Science,CS,3,FALL 2024,A
1,aa123,Alice Anderson,CS,CS225,Data Structures,CS,3,SPRING 2025,A-
2,bb234,Bob Brown,Math,MATH241,Calculus III,Math,3,Fall 2025,B+
3,cc345,Cathy Chen,CS,CS101,Intro to Computer Science,CS,3,Fall 2025,C+
4,cc345,Cathy Chen,CS,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A
5,dd456,David Davis,Economics,ECON102,Microeconomic Principles,Economics,3,Fall 2025,B
6,ee567,Emma Evans,Math,CS225,Data Structures,CS,3,Fall 2025,B
7,ee567,Emma Evans,Math,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A-


## Set up SQLite database

In [8]:
# Connect to (or create) a SQLite database file
conn = sqlite3.connect("class22_demo.db")

# Enable foreign key constraints
conn.execute("PRAGMA foreign_keys = ON;")

print("Database ready")

Database ready


## Create SQL tables

- Create three tables
    - Drop tables first if they already exist
- Requirements:
    - `student_profile: has autoincrement primary key student_id
    - `course_profile: has autoincrement primary key course_id
    - `registration: has autoincrement registration_id, and foreign keys student_id and course_id


In [10]:
# drop tables if exists
conn.execute("DROP TABLE IF EXISTS registration;")
conn.execute("DROP TABLE IF EXISTS student_profile;")
conn.execute("DROP TABLE IF EXISTS course_profile;")

sql_script = """
CREATE TABLE student_profile (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    net_id TEXT UNIQUE NOT NULL,
    full_name TEXT,
    major TEXT
);
CREATE TABLE course_profile (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_no TEXT UNIQUE NOT NULL,
    course_title TEXT NOT NULL,
    department TEXT,
    credits INTEGER
);
CREATE TABLE registration (
    registration_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    course_id INTEGER,
    term TEXT,
    grade TEXT,
    FOREIGN KEY (student_id) REFERENCES student_profile(student_id),
    FOREIGN KEY (course_id) REFERENCES course_profile(course_id)
);
"""
conn.executescript(sql_script)
print('Tables created')

Tables created


## Populate Data to SQLite Tables

### student_profile
- Extract student_profile related columns
- Remove duplicates
- Populate to DB table

In [13]:
df.columns

Index(['net_id', 'full_name', 'major', 'course_no', 'course_title',
       'department', 'credits', 'term', 'grade'],
      dtype='object')

In [14]:
df_student = df[['net_id', 'full_name', 'major']]
df_student

,net_id,full_name,major
0,aa123,Alice Anderson,CS
1,aa123,Alice Anderson,CS
2,bb234,Bob Brown,Math
3,cc345,Cathy Chen,CS
4,cc345,Cathy Chen,CS
5,dd456,David Davis,Economics
6,ee567,Emma Evans,Math
7,ee567,Emma Evans,Math


In [15]:
df_student = df_student.drop_duplicates()
df_student

,net_id,full_name,major
0,aa123,Alice Anderson,CS
2,bb234,Bob Brown,Math
3,cc345,Cathy Chen,CS
5,dd456,David Davis,Economics
6,ee567,Emma Evans,Math


In [16]:
df_student.to_sql('student_profile', conn, if_exists='append', index=False)
# Verify result
df_student = pd.read_sql_query('select * from student_profile', conn)
df_student

,student_id,net_id,full_name,major
0,1,aa123,Alice Anderson,CS
1,2,bb234,Bob Brown,Math
2,3,cc345,Cathy Chen,CS
3,4,dd456,David Davis,Economics
4,5,ee567,Emma Evans,Math


### course_profile
- Extract course_profile related columns
- Remove duplicates
- Populate to DB table

In [18]:
df.columns

Index(['net_id', 'full_name', 'major', 'course_no', 'course_title',
       'department', 'credits', 'term', 'grade'],
      dtype='object')

In [19]:
df_course = df[['course_no', 'course_title', 'department', 'credits']]
df_course

,course_no,course_title,department,credits
0,CS101,Intro to Computer Science,CS,3
1,CS225,Data Structures,CS,3
2,MATH241,Calculus III,Math,3
3,CS101,Intro to Computer Science,CS,3
4,STAT400,Statistics and Probability,Statistics,4
5,ECON102,Microeconomic Principles,Economics,3
6,CS225,Data Structures,CS,3
7,STAT400,Statistics and Probability,Statistics,4


In [20]:
df_course = df_course.drop_duplicates()
df_course

,course_no,course_title,department,credits
0,CS101,Intro to Computer Science,CS,3
1,CS225,Data Structures,CS,3
2,MATH241,Calculus III,Math,3
4,STAT400,Statistics and Probability,Statistics,4
5,ECON102,Microeconomic Principles,Economics,3


In [21]:
df_course.to_sql('course_profile', conn, if_exists='append', index=False)
# Verify result
df_course = pd.read_sql_query('select * from course_profile', conn)
df_course

,course_id,course_no,course_title,department,credits
0,1,CS101,Intro to Computer Science,CS,3
1,2,CS225,Data Structures,CS,3
2,3,MATH241,Calculus III,Math,3
3,4,STAT400,Statistics and Probability,Statistics,4
4,5,ECON102,Microeconomic Principles,Economics,3


### registration

- Each row in original dataset is a registration
- Add student_id and course_id to the dataframe
- Extract registration related columns
- Populate to DB table

In [23]:
df.columns

Index(['net_id', 'full_name', 'major', 'course_no', 'course_title',
       'department', 'credits', 'term', 'grade'],
      dtype='object')

In [24]:
df

,net_id,full_name,major,course_no,course_title,department,credits,term,grade
0,aa123,Alice Anderson,CS,CS101,Intro to Computer Science,CS,3,FALL 2024,A
1,aa123,Alice Anderson,CS,CS225,Data Structures,CS,3,SPRING 2025,A-
2,bb234,Bob Brown,Math,MATH241,Calculus III,Math,3,Fall 2025,B+
3,cc345,Cathy Chen,CS,CS101,Intro to Computer Science,CS,3,Fall 2025,C+
4,cc345,Cathy Chen,CS,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A
5,dd456,David Davis,Economics,ECON102,Microeconomic Principles,Economics,3,Fall 2025,B
6,ee567,Emma Evans,Math,CS225,Data Structures,CS,3,Fall 2025,B
7,ee567,Emma Evans,Math,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A-


In [25]:
df_all = pd.merge(left=df, right=df_student[['net_id', 'student_id']], on='net_id', how='left')
df_all = pd.merge(left=df_all, right=df_course[['course_no', 'course_id']], on='course_no', how='left')
df_all

,net_id,full_name,major,course_no,course_title,department,credits,term,grade,student_id,course_id
0,aa123,Alice Anderson,CS,CS101,Intro to Computer Science,CS,3,FALL 2024,A,1,1
1,aa123,Alice Anderson,CS,CS225,Data Structures,CS,3,SPRING 2025,A-,1,2
2,bb234,Bob Brown,Math,MATH241,Calculus III,Math,3,Fall 2025,B+,2,3
3,cc345,Cathy Chen,CS,CS101,Intro to Computer Science,CS,3,Fall 2025,C+,3,1
4,cc345,Cathy Chen,CS,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A,3,4
5,dd456,David Davis,Economics,ECON102,Microeconomic Principles,Economics,3,Fall 2025,B,4,5
6,ee567,Emma Evans,Math,CS225,Data Structures,CS,3,Fall 2025,B,5,2
7,ee567,Emma Evans,Math,STAT400,Statistics and Probability,Statistics,4,Fall 2025,A-,5,4


In [26]:
df_registration = df_all[['student_id', 'course_id', 'term', 'grade']]
df_registration

,student_id,course_id,term,grade
0,1,1,FALL 2024,A
1,1,2,SPRING 2025,A-
2,2,3,Fall 2025,B+
3,3,1,Fall 2025,C+
4,3,4,Fall 2025,A
5,4,5,Fall 2025,B
6,5,2,Fall 2025,B
7,5,4,Fall 2025,A-


In [27]:
df_registration.to_sql('registration', conn, if_exists='append', index=False)
# Verify result
df_registration = pd.read_sql_query('select * from registration', conn)
df_registration

,registration_id,student_id,course_id,term,grade
0,1,1,1,FALL 2024,A
1,2,1,2,SPRING 2025,A-
2,3,2,3,Fall 2025,B+
3,4,3,1,Fall 2025,C+
4,5,3,4,Fall 2025,A
5,6,4,5,Fall 2025,B
6,7,5,2,Fall 2025,B
7,8,5,4,Fall 2025,A-


## Commit and Close DB

In [29]:
conn.commit()
conn.close()

## Add New Registration

### Process to Add New Registrations:
- Check if student is already in student_profile, if not, add new student
- Check if course is already in course_profile, if not, add new course
- Add new registration

### Add New Registration to DB


In [31]:
conn = sqlite3.connect("class22_demo.db")

In [32]:
df_new_reg = pd.DataFrame({'full_name': ['Cathy Chen'], 'net_id':['cc345'], 'major':['CS'],
                           'course_no': ['BDI475'], 'course_title':['Introduction to Data Analytics Applications in Business'],
                           'credits':[4], 'term':['SPRING 2025'], 'grade':['A-']}) 
df_new_reg

,full_name,net_id,major,course_no,course_title,credits,term,grade
0,Cathy Chen,cc345,CS,BDI475,Introduction to Data Analytics Applications in...,4,SPRING 2025,A-


In [33]:
# Check if student already in DB
pd.read_sql_query("select * from student_profile where net_id='cc345'", conn)

,student_id,net_id,full_name,major
0,3,cc345,Cathy Chen,CS


In [34]:
# Check if course already in DB
pd.read_sql_query("select * from course_profile where course_no='BDI475'", conn)

,course_id,course_no,course_title,department,credits


In [35]:
# Add new course to DB
df_new_reg[['course_no', 'course_title', 'credits']].to_sql('course_profile', conn, if_exists='append', index=False)

1

In [36]:
pd.read_sql_query("select * from course_profile where course_no='BDI475'", conn)

,course_id,course_no,course_title,department,credits
0,6,BDI475,Introduction to Data Analytics Applications in...,None,4


In [37]:
# Add new registration
df_new_reg['student_id'] = 3
df_new_reg['course_id'] = 6
df_new_reg[['student_id', 'course_id', 'term', 'grade']].to_sql('registration', conn, if_exists='append', index=False)
conn.commit()

In [38]:
pd.read_sql_query('select * from registration', conn)

,registration_id,student_id,course_id,term,grade
0,1,1,1,FALL 2024,A
1,2,1,2,SPRING 2025,A-
2,3,2,3,Fall 2025,B+
3,4,3,1,Fall 2025,C+
4,5,3,4,Fall 2025,A
5,6,4,5,Fall 2025,B
6,7,5,2,Fall 2025,B
7,8,5,4,Fall 2025,A-
8,9,3,6,SPRING 2025,A-


## Explore Sqlite DB with DBrowser and DBeaver

- Install [DBeaver](https://dbeaver.io/download/)
- Open the DB in DBeaver
- Visualize ER Diagram


## Registration System ER Diagram
<img src="https://github.com/user-attachments/assets/d6d45764-92ea-436f-93c8-6680435b7a09 " width=600>

## Data Analysis

- Find all students who got an 'A' for a course.
- List student name, course no, course title, term and grade.

In [41]:
df_student = pd.read_sql_query('select * from student_profile', conn)
df_student

,student_id,net_id,full_name,major
0,1,aa123,Alice Anderson,CS
1,2,bb234,Bob Brown,Math
2,3,cc345,Cathy Chen,CS
3,4,dd456,David Davis,Economics
4,5,ee567,Emma Evans,Math


In [42]:
df_course = pd.read_sql_query('select * from course_profile', conn)
df_course

,course_id,course_no,course_title,department,credits
0,1,CS101,Intro to Computer Science,CS,3
1,2,CS225,Data Structures,CS,3
2,3,MATH241,Calculus III,Math,3
3,4,STAT400,Statistics and Probability,Statistics,4
4,5,ECON102,Microeconomic Principles,Economics,3
5,6,BDI475,Introduction to Data Analytics Applications in...,None,4


In [43]:
df_registration = pd.read_sql_query('select * from registration', conn)
df_registration

,registration_id,student_id,course_id,term,grade
0,1,1,1,FALL 2024,A
1,2,1,2,SPRING 2025,A-
2,3,2,3,Fall 2025,B+
3,4,3,1,Fall 2025,C+
4,5,3,4,Fall 2025,A
5,6,4,5,Fall 2025,B
6,7,5,2,Fall 2025,B
7,8,5,4,Fall 2025,A-
8,9,3,6,SPRING 2025,A-


In [44]:
df_all = pd.merge(left=df_registration,right=df_student,on='student_id', how='left')
df_all = pd.merge(left=df_all, right=df_course, on='course_id', how='left')
df_all.head(2)

,registration_id,student_id,course_id,term,grade,net_id,full_name,major,course_no,course_title,department,credits
0,1,1,1,FALL 2024,A,aa123,Alice Anderson,CS,CS101,Intro to Computer Science,CS,3
1,2,1,2,SPRING 2025,A-,aa123,Alice Anderson,CS,CS225,Data Structures,CS,3


In [45]:
df_all[df_all.grade=='A'][['full_name', 'course_no', 'course_title', 'grade']]

,full_name,course_no,course_title,grade
0,Alice Anderson,CS101,Intro to Computer Science,A
4,Cathy Chen,STAT400,Statistics and Probability,A


In [46]:
conn.close()